# Retrieval Quality Predictor Training

Train the retrieval quality regression model using **cross-encoder distillation** -- the cross-encoder is the accurate-but-slow "teacher," and the quality predictor is the fast "student."

**Key Deliverables:**
- Generate training data: 500 queries x 20 chunks = ~10K examples
- Cross-encoder scores as regression target (0-1)
- Train GradientBoostingRegressor with 5-fold CV
- SHAP explanations for feature importance
- MLflow tracking under `retrieval_quality` experiment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Data

Load eval queries and the chunk corpus for training data generation.

In [ ]:
eval_queries = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'eval_queries.parquet')
chunks = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'medical_chunks.parquet')

print(f"Eval queries: {len(eval_queries)}")
print(f"Chunks: {len(chunks)}")
eval_queries.head()

## 2. Generate Training Data via Cross-Encoder Distillation

For each eval query, retrieve top-20 chunks and score with the cross-encoder. Extract features per (query, chunk) pair to build the training set.

In [ ]:
import re
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from src.embeddings.vector_store import VectorStore
from src.retrieval.reranker import Reranker

store = VectorStore()
reranker = Reranker()

print("Models loaded.")

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Sample 500 queries
sample_queries = eval_queries.sample(n=min(500, len(eval_queries)), random_state=42)

training_records = []

for idx, row in sample_queries.iterrows():
    query = row['question']
    qtype = row.get('qtype', '')
    
    # Retrieve top-20 from vector store
    results = store.query(query, top_k=20)
    
    if not results:
        continue
    
    # Score with cross-encoder (teacher)
    pairs = [[query, r['text']] for r in results]
    ce_scores = cross_encoder.predict(pairs)
    
    # Normalize scores to 0-1
    ce_min, ce_max = ce_scores.min(), ce_scores.max()
    if ce_max > ce_min:
        ce_normalized = (ce_scores - ce_min) / (ce_max - ce_min)
    else:
        ce_normalized = np.full_like(ce_scores, 0.5)
    
    # Extract features for each (query, chunk) pair
    query_tokens = set(re.findall(r'\w+', query.lower()))
    
    for r, score_val in zip(results, ce_normalized):
        chunk_tokens = set(re.findall(r'\w+', r['text'].lower()))
        overlap = len(query_tokens & chunk_tokens) / max(len(query_tokens), 1)
        
        training_records.append({
            'cosine_similarity': r.get('score', 0.0),
            'bm25_score': r.get('rrf_score', 0.0),
            'token_overlap': overlap,
            'chunk_length': len(r['text']),
            'qtype_match': int(r.get('qtype', '') == qtype),
            'target': float(score_val),
        })

train_df = pd.DataFrame(training_records)
print(f"Training examples: {len(train_df)}")
train_df.describe()

## 3. Train Quality Predictor with MLflow

In [ ]:
import mlflow
from src.retrieval.quality_predictor import RetrievalQualityPredictor

mlflow.set_tracking_uri(str(PROJECT_ROOT / 'mlruns'))

feature_cols = ['cosine_similarity', 'bm25_score', 'token_overlap', 'chunk_length', 'qtype_match']
X = train_df[feature_cols].values
y = train_df['target'].values

predictor = RetrievalQualityPredictor()
predictor.train_and_log(X, y, experiment_name='retrieval_quality')

print("Training complete. Check MLflow UI for metrics.")

## 4. SHAP Explanations

Understand which retrieval signals drive quality predictions.

In [ ]:
import shap

explainer = shap.TreeExplainer(predictor.pipeline.named_steps['model'])

X_scaled = predictor.pipeline.named_steps['scaler'].transform(X)
shap_values = explainer.shap_values(X_scaled)

shap.summary_plot(shap_values, X_scaled, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'exports' / 'shap_quality_predictor.png'), dpi=150)
plt.show()
print("SHAP summary plot saved.")

## 5. Quality Threshold Analysis

Show how the predicted quality score maps to confidence:
- `predicted_quality < 0.4` -> surface "low confidence" warning
- `predicted_quality >= 0.4` -> generate answer normally

In [ ]:
predictions = predictor.pipeline.predict(X_scaled)
predictions_clipped = np.clip(predictions, 0, 1)

low_confidence = (predictions_clipped < 0.4).sum()
high_confidence = (predictions_clipped >= 0.4).sum()

print(f"Low confidence (< 0.4): {low_confidence} ({low_confidence/len(predictions_clipped):.1%})")
print(f"High confidence (>= 0.4): {high_confidence} ({high_confidence/len(predictions_clipped):.1%})")

plt.figure(figsize=(8, 4))
plt.hist(predictions_clipped, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(x=0.4, color='red', linestyle='--', label='Confidence threshold')
plt.xlabel('Predicted Quality Score')
plt.ylabel('Count')
plt.title('Distribution of Retrieval Quality Predictions')
plt.legend()
plt.tight_layout()
plt.show()